<img src='./src/dare.png' height=400> <img src='./src/Scared_Straight!.jpg' height = 400>

# D.A.R.E. and Scared Straight

- **D.A.R.E.:** police officers teach 5th and 6th graders to say no to drugs
- **Scared Straight:** kids already in trouble visit a prison, and inmates yell at them about prison life
- Both were tested in many trials
- Plan: bootstrap the ATE for each study, smallest to largest, then pool them
- Watch where **0** lands

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

rng = np.random.default_rng(5030)

## Bootstrap recap

- Resample each group with replacement, compute some statistic, repeat $R$ times
- Here we will compute the **average treatment effect**, the average difference in outcomes between two groups
- We will make sure to resample each group individually (don't pool them for resampling)
- As long as treatment is randomly assigned, there is no reason to think that the participants in group $A$ or $B$ are systematically different, and the **ATE** is an unbiased estimator of the true contrast in outcomes between groups $B$ and $A$. If the treatment isn't randomly assigned, we have to be careful with our interpretation.


In [ ]:
def ate(df, treat='dare', y='drug_use'):
    return df.loc[df[treat] == 1, y].mean() - df.loc[df[treat] == 0, y].mean()

def bootstrap(df, treat='dare', y='drug_use', R=1000):
    A = df[df[treat] == 1]   # treatment group
    B = df[df[treat] == 0]   # control group
    fx = []
    for r in range(R):
        A_r = A.sample(len(A), replace=True)
        B_r = B.sample(len(B), replace=True)
        fx.append(A_r[y].mean() - B_r[y].mean())
    return np.array(fx)

# Part 1: D.A.R.E.

- Each table has one row per kid: `dare` (1 = got D.A.R.E.) and `drug_use` (1 = yes)
- ATE = D.A.R.E. use rate − control use rate
- ATE $< 0$ means the D.A.R.E. kids used **less**

## Step 1: one small study

- Ahmed et al. (2002): 236 5th and 6th graders in Nashville
- Outcome: has the kid ever smoked?
- One survey: kids who had been through D.A.R.E. vs kids who hadn't

In [ ]:
ahmed = pd.read_csv('./data/dare_ahmed.csv')
pd.crosstab(ahmed['dare'], ahmed['drug_use'])

In [ ]:
fx_ahmed = bootstrap(ahmed, R=1000)
print('ATE:', ate(ahmed))
print('95% CI:', np.quantile(fx_ahmed, [.025, .975]))

sns.ecdfplot(fx_ahmed)
plt.axvline(0, color='black', linestyle='--')
plt.show()

<div class='alert alert-info'>

- If this were the only study you had read, what would you tell a school board?
- 88% of these kids had D.A.R.E. Who might the other 28 kids be?
- This was a survey, not a randomized trial. If D.A.R.E. had been randomly assigned, how might the results change?
- What is more relevant to the spread of this ECDF: n, the sample size; or R, the number of bootstraps?</div>


## Step 2: bigger studies

- Ringwalt et al. (1991): 20 North Carolina schools **randomly assigned** to D.A.R.E. or not
- 1,270 5th and 6th graders, surveyed right after the program

In [ ]:
ringwalt = pd.read_csv('./data/dare_ringwalt.csv')
fx_ringwalt = bootstrap(ringwalt)
print('ATE:', ate(ringwalt))
print('95% CI:', np.quantile(fx_ringwalt, [.025, .975]))

sns.ecdfplot(fx_ahmed, label='Ahmed (survey, n = 236)')
sns.ecdfplot(fx_ringwalt, label='Ringwalt (randomized, n = 1,270)')
plt.axvline(0, color='black', linestyle='--')
plt.legend()
plt.show()

<div class='alert alert-info'>

- Why are these two so different: the bigger sample, the randomization, or both?
 </div>

### All 11 studies, smallest to largest

In [ ]:
students = pd.read_csv('./data/dare_all_studies.csv')

boots = {}
results = []
for study, g in students.groupby('study'):
    boots[study] = bootstrap(g)
    L, U = np.quantile(boots[study], [.025, .975])
    results.append({'study': study, 'n': len(g), 'ATE': ate(g), 'L95': L, 'U95': U})

results = pd.DataFrame(results).sort_values('n')
results

In [ ]:
plt.errorbar(results['ATE'], results['study'],
             xerr=[results['ATE'] - results['L95'], results['U95'] - results['ATE']],
             fmt='o', capsize=3)
plt.axvline(0, color='black', linestyle='--')
plt.xlabel('ATE (< 0 = D.A.R.E. kids used less)')
plt.show()

- The studies **disagree**: from a 20-point drop (Ahmed) to a 3-point **increase** (Becker, the largest)
- Different kids, places, outcomes, and follow-up times, so each study has its own true effect
- The big effects come from the smallest studies, and Ahmed wasn't randomized

> Bigger n means your estimate is closer to the truth, and the bootstrap correctly reports that it's closer. The bootstrap describes the convergence; it doesn't cause it.

## Step 3: pool them

- Pooled ATE = average of the study ATEs, weighted by sample size
- Combining the $r$-th resample from every study gives the $r$-th pooled resample

In [ ]:
w = results['n'] / results['n'].sum()
pool = sum(w_k * boots[s] for w_k, s in zip(w, results['study']))
print('Pooled ATE:', (w * results['ATE']).sum())
print('95% CI:', np.quantile(pool, [.025, .975]))

for s in results['study']:
    sns.ecdfplot(boots[s], color='gray', alpha=0.4)
sns.ecdfplot(pool, color='C3', linewidth=3)
plt.axvline(0, color='black', linestyle='--')
plt.show()

- Centered right on **0**, and the 95% CI includes "D.A.R.E. did nothing"
- West & O'Neal's own pooled estimate: $d = 0.023$, 95% CI $(-0.04, 0.08)$

<div class='alert alert-info'>

- Most D.A.R.E. studies assigned **schools**, not students. Kids in the same school share officers, teachers, and friends. We resampled kids as if each one were independent. Should the real CIs be wider or narrower? Why?
- Does that change any conclusion above?</div>

# Part 2: Scared Straight

- 7 randomized trials (Petrosino et al., Campbell Collaboration review)
- Real counts this time: kids in each group, and how many reoffended
- Here kids were randomized **individually**, so resampling kids is right
- ATE = reoffending rate in program − reoffending rate in control, so ATE $> 0$ means the program made things **worse**

In [ ]:
ss = pd.read_csv('./data/ss_all_studies.csv')

ss_boots = {}
ss_results = []
for study, g in ss.groupby('study'):
    ss_boots[study] = bootstrap(g, treat='program', y='reoffend')
    L, U = np.quantile(ss_boots[study], [.025, .975])
    ss_results.append({'study': study, 'n': len(g), 'ATE': ate(g, 'program', 'reoffend'), 'L95': L, 'U95': U})

ss_results = pd.DataFrame(ss_results).sort_values('n')
ss_results

## Step 1: every study, smallest to largest

In [ ]:
plt.errorbar(ss_results['ATE'], ss_results['study'],
             xerr=[ss_results['ATE'] - ss_results['L95'], ss_results['U95'] - ss_results['ATE']],
             fmt='o', capsize=3, color='C1')
plt.axvline(0, color='black', linestyle='--')
plt.xlabel('ATE (> 0 = program made reoffending worse)')
plt.show()

<div class='alert alert-info'>Most of these CIs include 0, but every estimate is above 0.

- Before running the next cell: what will happen when we pool them?</div>

## Step 2: pool them

In [ ]:
w = ss_results['n'] / ss_results['n'].sum()
ss_pool = sum(w_k * ss_boots[s] for w_k, s in zip(w, ss_results['study']))
print('Pooled ATE:', (w * ss_results['ATE']).sum())
print('95% CI:', np.quantile(ss_pool, [.025, .975]))

for s in ss_results['study']:
    sns.ecdfplot(ss_boots[s], color='gray', alpha=0.4)
sns.ecdfplot(ss_pool, color='C3', linewidth=3)
plt.axvline(0, color='black', linestyle='--')
plt.show()

- The whole ECDF is to the **right** of 0: Scared Straight **increased** reoffending
- Seven "inconclusive" studies, all leaning the same way, add up to a clear answer


# Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.ecdfplot(pool, ax=axes[0], color='C0', linewidth=3)
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set_title('D.A.R.E., pooled (< 0 = helped)')
sns.ecdfplot(ss_pool, ax=axes[1], color='C1', linewidth=3)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Scared Straight, pooled (> 0 = hurt)')
plt.show()

## What did it cost?

- **D.A.R.E.:** about \$750 million a year, and \$1 to \$1.3 billion counting officers' time
- **Scared Straight:** about \$65 per kid, but **−\$4,949** in expected benefits per kid from extra crime and lower earnings
    - Each \$1 spent is associated with about \$76 in losses (Washington State Institute for Public Policy)

## How do we know when enough is enough?
- Stop when more data can't change what you'd do. Before the study, decide the **smallest effect** that would actually matter, like the break-even point where the program pays for itself. Once the confidence interval sits entirely on one side of that line, you have your answer.

- Cumulative meta-analysis: Line up every trial in the order it was published, and re-pool the results each time a new one comes in. The pooled estimate swings around at first and then settles down. Once it has settled on one side of the decision line, new trials stop changing the answer.

- Health economists make this a formal rule with **value-of-information analysis**. Instead of asking 'is it significant?', you ask: 'If we ran the next study, how much better would our decision get, on average? And is that worth what the study costs?' If the next trial is unlikely to change the decision, its value is low, and you shouldn't run it, even if it would shrink the confidence interval.

### Sources

- **D.A.R.E.:** West & O'Neal (2004), *American Journal of Public Health* 94(6):1027–1029, Table 1. Ahmed et al. (2002) uses its reported rates; the other studies are rebuilt from each study's $n$ and effect size, assuming a 20% control-group use rate (see `instructor_only/rebuild_study_data.py`)
- **Scared Straight:** Petrosino et al., Campbell Collaboration review (2004), forest-plot counts
- **Costs:** West & O'Neal (2004); Shepard, "The Economic Costs of D.A.R.E." (2001), via secondary sources; WSIPP benefit-cost summary (2012)
- Studies here are weighted by sample size, so pooled numbers differ slightly from the published ones